## 00 — Bounding Boxes

We have four simplified LOD files. Each still contains thousands of features spread across the globe. When a user is looking at Western Europe at zoom 8, there is no reason to send Siberian railroads to the renderer.

The first tool for eliminating invisible features is the **bounding box** — the smallest axis-aligned rectangle that fully contains a geometry.

This notebook covers:
1. What a bounding box is and how it is stored
2. How to compute one from a feature's coordinates
3. Why the railroad dataset already has them — and what to do with that

## What Is a Bounding Box?

An **axis-aligned bounding box (AABB)** is defined by four values:

```
[lon_min, lat_min, lon_max, lat_max]
```

This is also the GeoJSON `bbox` convention. Every GeoJSON object can optionally carry a `bbox` field with this exact format.

```
lat_max  ┌───────────────┐
         │               │
         │   feature     │
         │               │
lat_min  └───────────────┘
      lon_min          lon_max
```

The bounding box does not describe the shape of the feature — only its **extent**. Two very different shapes can have identical bounding boxes.

## The Railroad Dataset Already Has Bounding Boxes

Recall from Module 00 that each feature in `ne_10m_railroads.geojson` has a `bbox` key.

Let's inspect it.

In [1]:
import json
from pathlib import Path

data_path = Path("../../data/ne_10m_railroads.geojson")
with open(data_path) as f:
    railroads = json.load(f)

feature = railroads["features"][0]

print("Feature keys:", list(feature.keys()))
print("bbox:", feature["bbox"])
print()
print("Format: [lon_min, lat_min, lon_max, lat_max]")

Feature keys: ['type', 'properties', 'bbox', 'geometry']
bbox: [30.730275, 69.448054, 30.782502, 69.461111]

Format: [lon_min, lat_min, lon_max, lat_max]


The `bbox` field is precomputed and trustworthy for the raw data.

However, our LOD files were written by the pipeline in the previous module — without `bbox` fields. So we need to be able to **compute** a bounding box from coordinates ourselves.

## Computing a Bounding Box

Given a list of `[lon, lat]` coordinate pairs, the bounding box is simply the min and max of each axis.

In [2]:
def feature_bbox(feature):
    """
    Compute [lon_min, lat_min, lon_max, lat_max] for a GeoJSON LineString feature.
    """
    coords = feature["geometry"]["coordinates"]
    lons = [c[0] for c in coords]
    lats = [c[1] for c in coords]
    return [min(lons), min(lats), max(lons), max(lats)]

In [3]:
# Verify our result matches the precomputed bbox
computed  = feature_bbox(feature)
precomputed = feature["bbox"]

print("Computed:    ", computed)
print("Precomputed: ", precomputed)
print("Match:", computed == precomputed)

Computed:     [30.730275, 69.448054, 30.782502, 69.461111]
Precomputed:  [30.730275, 69.448054, 30.782502, 69.461111]
Match: True


## Visualizing a Feature and Its Bounding Box

Let's display one feature and its bounding box on a map to see what it looks like.

In [4]:
from ipyleaflet import Map, GeoJSON

# Pick a longer feature for a more interesting bbox
long_features = sorted(railroads["features"], key=lambda f: len(f["geometry"]["coordinates"]), reverse=True)
f = long_features[2]

bbox = feature_bbox(f)
lon_min, lat_min, lon_max, lat_max = bbox

# Build the bbox as a GeoJSON polygon
bbox_polygon = {
    "type": "Feature",
    "properties": {"name": "bounding box"},
    "geometry": {
        "type": "Polygon",
        "coordinates": [[
            [lon_min, lat_min],
            [lon_max, lat_min],
            [lon_max, lat_max],
            [lon_min, lat_max],
            [lon_min, lat_min],
        ]]
    }
}

center_lat = (lat_min + lat_max) / 2
center_lon = (lon_min + lon_max) / 2

m = Map(center=[center_lat, center_lon], zoom=5)

m.add(GeoJSON(data={"type": "FeatureCollection", "features": [f]},
              style={"color": "#cc3300", "weight": 2}))
m.add(GeoJSON(data={"type": "FeatureCollection", "features": [bbox_polygon]},
              style={"color": "#0066cc", "weight": 1.5, "fillOpacity": 0.05}))
m

Map(center=[63.0397215, 75.576944], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_title'…

## Bounding Boxes for the LOD Files

Our LOD output files do not have precomputed `bbox` fields. We will compute them on the fly during culling.

As an optimization preview: we could precompute and store bounding boxes once at pipeline time, then just read the stored values during culling. This is a common real-world pattern.

For now, let's verify the function works on a LOD feature.

In [5]:
lod_path = Path("../../data/lod/railroads_fine.geojson")
with open(lod_path) as f:
    fine = json.load(f)

sample = fine["features"][100]
bbox = feature_bbox(sample)

print("LOD feature bbox:", bbox)
print("Coordinate count:", len(sample["geometry"]["coordinates"]))

LOD feature bbox: [66.311406, 66.714926, 68.935817, 68.190447]
Coordinate count: 26


## Exercise A

Write a function `collection_bbox(features)` that returns the bounding box of an **entire FeatureCollection** — the smallest rectangle that contains all features.

Apply it to each of the four LOD files and compare the results. Do they all cover the same geographic extent?

In [6]:
# Write collection_bbox(features) and apply to all four LOD files

def collection_bbox(features):
    """
    Return [lon_min, lat_min, lon_max, lat_max] for an entire list of
    GeoJSON LineString features.
    """
    if not features:
        return None

    boxes = [feature_bbox(f) for f in features]
    lon_min = min(b[0] for b in boxes)
    lat_min = min(b[1] for b in boxes)
    lon_max = max(b[2] for b in boxes)
    lat_max = max(b[3] for b in boxes)
    return [lon_min, lat_min, lon_max, lat_max]

lod_dir = Path("../../data/lod")
lod_files = {
    "coarse":     "railroads_coarse.geojson",
    "medium":     "railroads_medium.geojson",
    "fine":       "railroads_fine.geojson",
    "extra_fine": "railroads_extra_fine.geojson",
}

collection_bboxes = {}

for level, filename in lod_files.items():
    with open(lod_dir / filename) as f:
        fc = json.load(f)

    bbox = collection_bbox(fc["features"])
    collection_bboxes[level] = bbox
    rounded = [round(v, 4) for v in bbox]
    print(f"{level:<12} {rounded}")

print()
reference = collection_bboxes["fine"]
for level, bbox in collection_bboxes.items():
    same_as_fine = all(abs(a - b) < 1e-9 for a, b in zip(bbox, reference))
    print(f"{level:<12} same exact extent as fine? {same_as_fine}")

print("\nConclusion:")
print("The LOD files should cover nearly the same world area, but they may not be exactly identical.")
print("The coarse level uses filtering and all levels use simplification, so some extreme points/features can disappear.")


coarse       [-123.0147, -41.4752, 150.9617, 60.9765]
medium       [-150.1122, -51.8947, 179.3578, 69.6044]
fine         [-150.1122, -51.8947, 179.3578, 69.6044]
extra_fine   [-150.1122, -51.8953, 179.3578, 69.6044]

coarse       same exact extent as fine? False
medium       same exact extent as fine? True
fine         same exact extent as fine? True
extra_fine   same exact extent as fine? False

Conclusion:
The LOD files should cover nearly the same world area, but they may not be exactly identical.
The coarse level uses filtering and all levels use simplification, so some extreme points/features can disappear.


## Exercise B

Find the **5 features with the largest bounding box area** in the fine LOD file.

Bounding box area = `(lon_max - lon_min) * (lat_max - lat_min)`.

Print each one's bbox area and its `category` property. Do the results make geographic sense?

In [7]:
# Find the 5 features with the largest bounding box area in railroads_fine.geojson

def bbox_area(bbox):
    lon_min, lat_min, lon_max, lat_max = bbox
    return (lon_max - lon_min) * (lat_max - lat_min)

largest = []
for f in fine["features"]:
    bbox = feature_bbox(f)
    area = bbox_area(bbox)
    largest.append((area, bbox, f))

largest = sorted(largest, key=lambda item: item[0], reverse=True)[:5]

print(f"{'Rank':<6} {'Area':>12} {'Category':<25} {'BBox'}")
print("-" * 85)

for rank, (area, bbox, f) in enumerate(largest, start=1):
    props = f.get("properties", {})
    category = props.get("category", "unknown")
    print(f"{rank:<6} {area:>12.2f} {category:<25} {[round(v, 3) for v in bbox]}")

print("\nConclusion:")
print("Large bbox areas usually belong to long railroad features that stretch far east-west and/or north-south.")
print("That makes geographic sense because a longer route naturally creates a larger enclosing rectangle.")


Rank           Area Category                  BBox
-------------------------------------------------------------------------------------
1             29.31 0                         [90.609, 29.645, 94.943, 36.409]
2             18.65 0                         [132.256, -23.55, 134.323, -14.531]
3             17.70 3                         [-64.094, -26.192, -58.157, -23.211]
4             17.39 2                         [14.022, 54.803, 20.0, 57.711]
5             12.35 2                         [-49.138, -5.492, -44.372, -2.899]

Conclusion:
Large bbox areas usually belong to long railroad features that stretch far east-west and/or north-south.
That makes geographic sense because a longer route naturally creates a larger enclosing rectangle.


## Check Your Understanding

Two different railroad features can have identical bounding boxes even though they follow completely different paths.

Describe a scenario where this happens — what would the two features look like? And does this cause any problem for our culling system?

---

### Answer

Two railroad features could have the same bounding box if they share the same minimum and maximum longitude and latitude, even if their paths are different. For example, one railroad could run diagonally from the southwest corner to the northeast corner of the box, while another could curve around the outside edges of the same box.

This does not break the culling system because the bounding box test is only a fast first filter. It may keep a few extra features that are not actually visible, but it will not incorrectly remove a feature whose bounding box is outside the viewport. For rendering, this kind of false positive is usually acceptable because it is much faster than testing the exact line geometry every time.


## Next

In [01 — Intersection Test](./01-Intersection_Test.ipynb), we write the function that checks whether a feature's bounding box overlaps the current viewport.